# Migrating from Document AI to AI_EXTRACT

This demo shows how to migrate from the deprecated **Document AI** (`model!PREDICT`) to the new **AI_EXTRACT** function.

### Why Migrate?
- Document AI and `model!PREDICT` are **deprecated** (BCR-2156)
- AI_EXTRACT is **zero-shot** — no model training required
- Powered by **Arctic-extract** vision model
- Part of the broader **AISQL** function family

### Migration Comparison
| Document AI (Old) | AI_EXTRACT (New) |
|-------------------|------------------|
| Model creation in Snowsight UI | No setup needed |
| Fine-tuning required for accuracy | Zero-shot, ready to use |
| `model!PREDICT(GET_PRESIGNED_URL(...))` | `AI_EXTRACT(file => TO_FILE(...), ...)` |
| Arctic-TILT model | Arctic-extract vision model |
| Confidence scores | Flexible response formats |

## Setup: Create Database, Schema, and Stage

In [ ]:
USE DATABASE DOC_AI_DEPRECATION;
USE SCHEMA PUBLIC;

In [ ]:
CREATE OR REPLACE STAGE DEPRECATED_DOC_AI_IMAGES
    DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

CREATE OR REPLACE STAGE DEMO_DOCS
    DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

MY_STAGE = 'DEMO_DOCS'
MY_FILE_NAME = "data/*"

put_result = session.file.put(MY_FILE_NAME, MY_STAGE, auto_compress=False, overwrite=True)

In [ ]:
ALTER STAGE DEMO_DOCS REFRESH;
SELECT * FROM DIRECTORY(@DEMO_DOCS);

---
## The Old Way: Document AI with model!PREDICT

**Problems with the old approach:**
- Required creating and training a model in Snowsight UI
- Fine-tuning needed for good accuracy
- Model management overhead
- Limited flexibility in question formats

In [ ]:
-- SELECT 
--     DOC_AI_DEPRECATION.PUBLIC.DEPRECATION_DEMO!PREDICT(
--         GET_PRESIGNED_URL(@DEMO_DOCS, relative_path),
--         1
--     ) AS predictions
-- FROM DIRECTORY(@DEMO_DOCS)
-- LIMIT 1;

---
## Migration Path: Preserving Your Fine-Tuned Model

If you have a fine-tuned Document AI model you want to preserve, follow these steps:

**TODO: Add instructions here**
1. Step 1: ...
2. Step 2: ...
3. Step 3: ...

In [ ]:
CREATE OR REPLACE FILE FORMAT my_json
  TYPE = 'JSON';

In [ ]:
CREATE OR REPLACE TABLE exported_data_table AS (
   SELECT
      input_file.$1:file AS file,
      input_file.$1:prompt AS prompt,
      input_file.$1:annotatedResponse AS response
   FROM '@DOC_AI_DEPRECATION.PUBLIC.DEPRECATED_DOC_AI_IMAGES/DOC_AI_DEPRECATION_PUBLIC_DEPRECATION_DEMO_2026_01_20_19_36_40/annotations.jsonl' (FILE_FORMAT => my_json) input_file
   WHERE response != '{}'
);

In [ ]:
SELECT
*
FROM EXPORTED_DATA_TABLE;

In [ ]:
INSERT INTO PROMPT_TEMPLATES
WITH PROMPT as (
    SELECT
    PROMPT
    FROM EXPORTED_DATA_TABLE
    LIMIT 1
)
SELECT
'INSPECTION_REVIEWS',
PARSE_JSON()


In [ ]:
SELECT AI_EXTRACT(
    MODEL         => 'DEPRECATION_DEMO',
    FILE          => '@DEMO_DOCS.Manual_2022-02-01.pdf',
    RESPONSEFORMAT => ,
    CONFIG        => {'output_details': TRUE}
) AS result;

---
## The New Way: AI_EXTRACT

AI_EXTRACT is a **zero-shot** function — no model training required. Just ask questions!

In [ ]:
CREATE TABLE IF NOT EXISTS prompt_templates (
    template_id VARCHAR PRIMARY KEY,
    response_format VARIANT
);

### Example 1: Basic Extraction — Single File

In [ ]:
-- Extract specific fields from a single document
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO_DOCS', 'invoices/invoice_001.pdf'),
    responseFormat => [
        ['invoice_number', 'What is the invoice number?'],
        ['date', 'What is the invoice date?'],
        ['total', 'What is the total amount due?']
    ]
) AS extracted_data;

### Example 2: Response Format — Object Syntax

In [ ]:
-- Alternative: Use object syntax for responseFormat
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO_DOCS', 'invoices/invoice_001.pdf'),
    responseFormat => {
        'vendor': 'Who is the vendor or seller?',
        'customer': 'Who is the customer or buyer?',
        'payment_terms': 'What are the payment terms?'
    }
) AS extracted_data;

### Example 3: Response Format — Simple Array of Questions

In [ ]:
-- Simple array: Labels are auto-generated
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO_DOCS', 'invoices/invoice_001.pdf'),
    responseFormat => [
        'What is the invoice number?',
        'What is the total amount?',
        'What is the due date?'
    ]
) AS extracted_data;

### Example 4: Parsing the JSON Response

In [ ]:
-- Parse the JSON response into columns
SELECT
    json_data:response.invoice_number::STRING AS invoice_number,
    json_data:response.date::STRING AS invoice_date,
    json_data:response.total::STRING AS total_amount,
    json_data:response.vendor::STRING AS vendor_name
FROM (
    SELECT AI_EXTRACT(
        file => TO_FILE('@DEMO_DOCS', 'invoices/invoice_001.pdf'),
        responseFormat => [
            ['invoice_number', 'What is the invoice number?'],
            ['date', 'What is the invoice date?'],
            ['total', 'What is the total amount?'],
            ['vendor', 'Who is the vendor?']
        ]
    ) AS json_data
);

### Example 5: Batch Processing — All Files in Directory

In [ ]:
-- Process all invoices in a directory
SELECT
    relative_path,
    json_data:response.invoice_number::STRING AS invoice_number,
    json_data:response.date::STRING AS invoice_date,
    json_data:response.total::STRING AS total_amount
FROM (
    SELECT
        relative_path,
        AI_EXTRACT(
            file => TO_FILE('@DEMO_DOCS', relative_path),
            responseFormat => [
                ['invoice_number', 'What is the invoice number?'],
                ['date', 'What is the invoice date?'],
                ['total', 'What is the total amount due?']
            ]
        ) AS json_data
    FROM DIRECTORY(@DEMO_DOCS)
    WHERE relative_path LIKE 'invoices/%.pdf'
);

### Example 6: Classification via Prompts

In [ ]:
-- Use AI_EXTRACT for yes/no classification questions
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO_DOCS', 'invoices/invoice_001.pdf'),
    responseFormat => [
        ['has_signature', 'Does this document have a signature? Answer Yes or No'],
        ['is_paid', 'Does this invoice show as paid? Answer Yes or No'],
        ['has_late_fee', 'Is there a late fee mentioned? Answer Yes or No'],
        ['document_type', 'What type of document is this: Invoice, Receipt, Quote, or Other?']
    ]
) AS classification;

### Example 7: Table Extraction with JSON Schema

In [ ]:
-- Extract tabular data (line items) using JSON schema
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO_DOCS', 'invoices/invoice_001.pdf'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'line_items': {
                    'description': 'Line items from the invoice',
                    'type': 'object',
                    'column_ordering': ['description', 'quantity', 'unit_price', 'amount'],
                    'properties': {
                        'description': {
                            'description': 'Item description',
                            'type': 'array'
                        },
                        'quantity': {
                            'description': 'Quantity',
                            'type': 'array'
                        },
                        'unit_price': {
                            'description': 'Unit price',
                            'type': 'array'
                        },
                        'amount': {
                            'description': 'Line total',
                            'type': 'array'
                        }
                    }
                }
            }
        }
    }
) AS table_data;

### Example 8: Array Extraction

In [ ]:
-- Extract a list of items as an array
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO_DOCS', 'resumes/resume_001.pdf'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'skills': {
                    'description': 'List all technical skills mentioned',
                    'type': 'array'
                },
                'companies': {
                    'description': 'List all company names where the person worked',
                    'type': 'array'
                },
                'certifications': {
                    'description': 'List all certifications or credentials',
                    'type': 'array'
                }
            }
        }
    }
) AS arrays_data;

### Example 9: Mixed Extraction (Entity + Table + Array)

In [ ]:
-- Extract different types in a single call
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO_DOCS', 'invoices/invoice_001.pdf'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'invoice_number': {
                    'description': 'Invoice number',
                    'type': 'string'
                },
                'total_amount': {
                    'description': 'Total amount due',
                    'type': 'string'
                },
                'payment_methods': {
                    'description': 'Accepted payment methods',
                    'type': 'array'
                },
                'line_items': {
                    'description': 'Invoice line items',
                    'type': 'object',
                    'column_ordering': ['item', 'qty', 'price'],
                    'properties': {
                        'item': {'type': 'array'},
                        'qty': {'type': 'array'},
                        'price': {'type': 'array'}
                    }
                }
            }
        }
    }
) AS mixed_data;

### Example 10: Create a Table from Extracted Data

In [ ]:
-- Create a structured table from document extraction
CREATE OR REPLACE TABLE extracted_invoices AS
SELECT
    relative_path AS source_file,
    json_data:response.invoice_number::STRING AS invoice_number,
    json_data:response.vendor::STRING AS vendor,
    json_data:response.customer::STRING AS customer,
    json_data:response.date::STRING AS invoice_date,
    json_data:response.due_date::STRING AS due_date,
    json_data:response.subtotal::STRING AS subtotal,
    json_data:response.tax::STRING AS tax,
    json_data:response.total::STRING AS total,
    CURRENT_TIMESTAMP() AS extracted_at
FROM (
    SELECT
        relative_path,
        AI_EXTRACT(
            file => TO_FILE('@DEMO_DOCS', relative_path),
            responseFormat => [
                ['invoice_number', 'Invoice number'],
                ['vendor', 'Vendor or seller name'],
                ['customer', 'Customer or buyer name'],
                ['date', 'Invoice date'],
                ['due_date', 'Payment due date'],
                ['subtotal', 'Subtotal before tax'],
                ['tax', 'Tax amount'],
                ['total', 'Total amount due']
            ]
        ) AS json_data
    FROM DIRECTORY(@DEMO_DOCS)
    WHERE relative_path LIKE 'invoices/%.pdf'
);

SELECT * FROM extracted_invoices;

### Example 11: Extract from Text (No File Needed)

In [ ]:
-- AI_EXTRACT also works on plain text
SELECT AI_EXTRACT(
    text => 'Invoice #12345 dated January 15, 2026. Bill to: Acme Corp, 123 Main St, Seattle WA 98101. Amount due: $5,432.10. Payment terms: Net 30.',
    responseFormat => [
        ['invoice_number', 'Invoice number'],
        ['customer', 'Customer name'],
        ['address', 'Customer address'],
        ['amount', 'Amount due'],
        ['terms', 'Payment terms']
    ]
) AS extracted;

### Example 12: Different Document Types

In [ ]:
-- Inspection Reports
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO_DOCS', 'reports/inspection_001.pdf'),
    responseFormat => [
        ['inspector', 'Who performed the inspection?'],
        ['date', 'Date of inspection'],
        ['location', 'Location or site inspected'],
        ['result', 'Pass or Fail?'],
        ['issues', 'What issues were found?'],
        ['recommendations', 'What recommendations were made?']
    ]
) AS inspection_data;

In [ ]:
-- Contracts
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO_DOCS', 'contracts/contract_001.pdf'),
    responseFormat => [
        ['parties', 'Who are the parties to this agreement?'],
        ['effective_date', 'When does this agreement become effective?'],
        ['term', 'What is the term or duration?'],
        ['value', 'What is the contract value or amount?'],
        ['termination', 'What are the termination conditions?']
    ]
) AS contract_data;

In [ ]:
-- Resumes
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO_DOCS', 'resumes/resume_001.pdf'),
    responseFormat => [
        ['name', 'Candidate full name'],
        ['email', 'Email address'],
        ['phone', 'Phone number'],
        ['location', 'City and state'],
        ['current_title', 'Current or most recent job title'],
        ['years_experience', 'Total years of experience']
    ]
) AS resume_data;

---
## AI_EXTRACT Capabilities Summary

| Feature | Details |
|---------|--------|
| **Supported Formats** | PDF, PNG, JPG, DOCX, PPTX, HTML, TXT, TIFF, BMP, GIF, WEBP, MD, EML |
| **Max File Size** | 100 MB |
| **Max Pages** | 125 pages |
| **Max Questions** | 100 entity extractions, 10 table extractions per call |
| **Response Formats** | Array of arrays, Object, Array of strings, JSON Schema |
| **Languages** | 20+ including English, Spanish, French, German, Chinese, Japanese, Arabic, etc. |
| **Model** | Arctic-extract (vision model — no OCR preprocessing needed) |

---
## Migration Checklist

1. **Identify your Document AI model builds** in Snowsight
2. **Review the questions** defined in each model build
3. **Convert to AI_EXTRACT format:**
   - Simple questions → Array of arrays `[['label', 'question'], ...]`
   - Table extraction → JSON Schema with `column_ordering`
4. **Update your pipelines:**
   - Replace `model!PREDICT(GET_PRESIGNED_URL(...))` with `AI_EXTRACT(file => TO_FILE(...), ...)`
5. **Test thoroughly** — zero-shot may behave differently than fine-tuned models
6. **Remove Document AI model builds** after successful migration

# Image Table Extraction Examples

The following examples demonstrate AI_EXTRACT's ability to extract structured data from images of tables. We'll process 4 different table images, each with varying complexity.

In [ ]:
MY_FILE_NAME = "data/*.jpg"
put_result = session.file.put(MY_FILE_NAME, MY_STAGE, auto_compress=False, overwrite=True)
put_result

In [ ]:
ALTER STAGE DEMO_DOCS REFRESH;

## Table 1: Simple Gene Table (CpG Sites)

A straightforward 4-column table with gene names, numeric values, and text descriptions.

In [ ]:
from PIL import Image
import os

img_path = os.path.join(os.getcwd(), "data", "simple_table_data.jpg")
img = Image.open(img_path)
st.image(img, caption="Simple Table: CpG Sites and Genes of Interest")

In [ ]:
SELECT AI_EXTRACT(
    TO_FILE('@DEMO_DOCS', 'simple_table_data.jpg'),
    {
        'gene_data': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'gene': {'type': 'string'},
                    'case': {'type': 'number'},
                    'control': {'type': 'number'},
                    'ipa_network': {'type': 'string'}
                }
            },
            'column_ordering': ['gene', 'case', 'control', 'ipa_network']
        }
    }
):gene_data AS extracted_genes;

## Table 2: Dual-Column Scientific Table (Gene Sequence Identity)

A table with gene identifiers and multiple numeric columns showing identity percentages and KaKs values.

In [ ]:
img_path = os.path.join(os.getcwd(), "data", "dual_column_table_data.jpg")
img = Image.open(img_path)
st.image(img, caption="Dual Column Table: Gene Sequence Identities and KaKs Values")

In [ ]:
SELECT AI_EXTRACT(
    TO_FILE('@DEMO_DOCS', 'dual_column_table_data.jpg'),
    {
        'sequence_data': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'gene': {'type': 'string'},
                    'identity_rice_maize': {'type': 'number'},
                    'identity_rice_sorghum': {'type': 'number'},
                    'identity_sorghum_maize': {'type': 'number'},
                    'kaks_rice_maize': {'type': 'number'},
                    'kaks_rice_sorghum': {'type': 'number'},
                    'kaks_sorghum_maize': {'type': 'number'},
                    'size_bp': {'type': 'integer'}
                }
            },
            'column_ordering': ['gene', 'identity_rice_maize', 'identity_rice_sorghum', 'identity_sorghum_maize', 'kaks_rice_maize', 'kaks_rice_sorghum', 'kaks_sorghum_maize', 'size_bp']
        }
    }
):sequence_data AS extracted_sequences;

## Table 3: Multi-Layer Nested Table (Medical Demographics)

A complex table with hierarchical categories (Age group, FIGO, Morphology, Surgery, Radiotherapy) and multiple population columns.

In [ ]:
img_path = os.path.join(os.getcwd(), "data", "multi_layer_nested_table_data.jpg")
img = Image.open(img_path)
st.image(img, caption="Multi-Layer Table: Tumor Characteristics by Population")

In [ ]:
SELECT AI_EXTRACT(
    TO_FILE('@DEMO_DOCS', 'multi_layer_nested_table_data.jpg'),
    {
        'tumor_characteristics': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'category': {'type': 'string'},
                    'subcategory': {'type': 'string'},
                    'philippine_freq': {'type': 'integer'},
                    'philippine_pct': {'type': 'number'},
                    'filipino_american_freq': {'type': 'integer'},
                    'filipino_american_pct': {'type': 'number'},
                    'caucasian_freq': {'type': 'integer'},
                    'caucasian_pct': {'type': 'number'},
                    'p_value': {'type': 'string'}
                }
            },
            'column_ordering': ['category', 'subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct', 'p_value']
        }
    }
):tumor_characteristics AS extracted_tumor_data;

## Table 4: Free-Text Rich Table (Systematic Review)

A literature review table with mixed content: citations, study designs, sample sizes, and lengthy outcome descriptions.

In [ ]:
img_path = os.path.join(os.getcwd(), "data", "free_text_table_data.jpg")
img = Image.open(img_path)
st.image(img, caption="Free-Text Table: Systematic Review of Vitamin D Studies")

In [ ]:
SELECT AI_EXTRACT(
    TO_FILE('@DEMO_DOCS', 'free_text_table_data.jpg'),
    {
        'systematic_review': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'author': {'type': 'string'},
                    'year': {'type': 'integer'},
                    'design_of_studies': {'type': 'string'},
                    'number_of_studies': {'type': 'integer'},
                    'sample_size': {'type': 'integer'},
                    'meta_analysis': {'type': 'string'},
                    'outcome': {'type': 'string'},
                    'association': {'type': 'string'}
                }
            },
            'column_ordering': ['author', 'year', 'design_of_studies', 'number_of_studies', 'sample_size', 'meta_analysis', 'outcome', 'association']
        }
    }
):systematic_review AS extracted_review_data;